<a href="https://colab.research.google.com/github/Gregory-lab-eng/python_Vistula/blob/main/Keras_breed_klassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install fastai -q

In [ ]:
from fastai.vision.all import untar_data, URLs
path = untar_data(URLs.PETS)
print(path)

<div><progress max="811706944" value="811712512"></progress> 100.00% [811712512/811706944 00:15&lt;00:00]</div>

/root/.fastai/data/oxford-iiit-pet


In [ ]:
"""
Pet Breed Classifier — Keras/TensorFlow
"""

import os
import re
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pathlib import Path
from sklearn.model_selection import train_test_split



In [ ]:
# ── 1. CONSTANTS ────────────────────────────────────────────────────────────
IMAGE_SIZE   = (224, 224)
RESIZE_SIZE  = (460, 460)   # mirrors item_tfms=Resize(460)
BATCH_SIZE   = 64
SEED         = 42

In [ ]:
# ── 2. LOAD FILE PATHS & LABELS ─────────────────────────────────────────────
# fastai does: untar_data(URLs.PETS) → typically ends up at ~/.fastai/data/oxford-iiit-pet/images
DATA_DIR = Path(os.path.expanduser("~/.fastai/data/oxford-iiit-pet/images"))

LABEL_RE = re.compile(r"(.+)_\d+\.jpg$", re.IGNORECASE)

def get_label(fname: Path) -> str | None:
    m = LABEL_RE.match(fname.name)
    return m.group(1) if m else None

all_files  = sorted(DATA_DIR.glob("*.jpg"))
all_labels = [get_label(f) for f in all_files]

# Drop files that don't match the pattern
pairs = [(f, l) for f, l in zip(all_files, all_labels) if l is not None]
files, labels = zip(*pairs)

# Integer-encode labels
unique_labels = sorted(set(labels))
label2idx     = {l: i for i, l in enumerate(unique_labels)}
num_classes   = len(unique_labels)
print(f"Found {len(files)} images across {num_classes} breeds.")

y = [label2idx[l] for l in labels]

# Train / validation split  (mirrors RandomSplitter(seed=42) → ~80/20)
train_files, val_files, train_y, val_y = train_test_split(
    list(files), y, test_size=0.2, random_state=SEED, stratify=y
)

Found 7390 images across 37 breeds.


In [ ]:
# ── 3. tf.data PIPELINE ──────────────────────────────────────────────────────
# Augmentation layers (mirrors aug_transforms(size=224, min_scale=0.75))
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom((-0.25, 0.0)),          # min_scale=0.75 → max zoom-out 25 %
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
], name="augmentation")

def load_and_preprocess(path, label, training=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    # item_tfms: resize to 460 first (keeps aspect ratio centre-crop style)
    img = tf.image.resize(img, RESIZE_SIZE)
    if training:
        # random crop to 224×224  (batch_tfms equivalent)
        img = tf.image.random_crop(img, [IMAGE_SIZE[0], IMAGE_SIZE[1], 3])
    else:
        img = tf.image.resize_with_crop_or_pad(img, IMAGE_SIZE[0], IMAGE_SIZE[1])
    # Normalise for ResNet (ImageNet stats)
    img = keras.applications.resnet.preprocess_input(img)
    return img, label

def make_dataset(file_list, label_list, training=False):
    paths  = [str(f) for f in file_list]
    ds = tf.data.Dataset.from_tensor_slices((paths, label_list))
    ds = ds.shuffle(len(paths), seed=SEED) if training else ds
    ds = ds.map(
        lambda p, l: load_and_preprocess(p, l, training=training),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    if training:
        ds = ds.map(
            lambda x, y: (augment(x, training=True), y),
            num_parallel_calls=tf.data.AUTOTUNE
        )
    return ds

train_ds = make_dataset(train_files, train_y, training=True)
val_ds   = make_dataset(val_files,   val_y,   training=False)

In [ ]:
# ── 4. MODEL — ResNet50 base + custom head ───────────────────────────────────

base = keras.applications.ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
)
base.trainable = False   # freeze — equivalent to the "head-only" first epoch pass

inputs = keras.Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.25)(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = keras.Model(inputs, outputs)
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 2048)           │         8,192 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,049,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 37)             │        18,981 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,666,021 (94.09 MB)

 Trainable params: 1,073,189 (4.09 MB)

 Non-trainable params: 23,592,832 (90.00 MB)

In [ ]:
# ── PHASE 1: Train head only (2-3 epochs) ──────────────────────────
base.trainable = False
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.fit(train_ds, validation_data=val_ds, epochs=3)

# ── PHASE 2: Unfreeze top layers and fine-tune ──────────────────────
base.trainable = True

# Only unfreeze the last N layers (start conservative)
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # 100x smaller LR — critical
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2)
    ]
)


Epoch 1/3
93/93 ━━━━━━━━━━━━━━━━━━━━ 122s 1s/step - accuracy: 0.2634 - loss: 3.1287 - val_accuracy: 0.7253 - val_loss: 0.8918
Epoch 2/3
93/93 ━━━━━━━━━━━━━━━━━━━━ 93s 988ms/step - accuracy: 0.3836 - loss: 2.3320 - val_accuracy: 0.7815 - val_loss: 0.7475
Epoch 3/3
93/93 ━━━━━━━━━━━━━━━━━━━━ 89s 953ms/step - accuracy: 0.4168 - loss: 2.1380 - val_accuracy: 0.7659 - val_loss: 0.7309
Epoch 1/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 128s 1s/step - accuracy: 0.4288 - loss: 2.0743 - val_accuracy: 0.7686 - val_loss: 0.7320 - learning_rate: 1.0000e-05
Epoch 2/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 90s 959ms/step - accuracy: 0.4481 - loss: 1.9736 - val_accuracy: 0.7673 - val_loss: 0.7409 - learning_rate: 1.0000e-05
Epoch 3/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - accuracy: 0.4555 - loss: 1.9670 - val_accuracy: 0.7740 - val_loss: 0.7211 - learning_rate: 1.0000e-05
Epoch 4/15
93/93 ━━━━━━━━━━━━━━━━━━━━ 101s 1s/step - accuracy: 0.4682 - loss: 1.9027 - val_accuracy: 0.7821 - val_loss: 0.7146 - learning_rate: 1.0000e

In [ ]:
# ── 6. SAVE ──────────────────────────────────────────────────────────────────
model.save("pet_breed_classifier.keras")
print("Model saved → pet_breed_classifier.keras")

# Optional: save label mapping for inference
import json
with open("label_map.json", "w") as f:
    json.dump({str(v): k for k, v in label2idx.items()}, f, indent=2)
print("Label map saved → label_map.json")

Model saved → pet_breed_classifier.keras
Label map saved → label_map.json


In [ ]:
!pip install gradio -q

In [ ]:
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
import gradio as gr
from PIL import Image

# ── 1. LOAD MODEL & LABEL MAP ────────────────────────────────────────────────
model = keras.models.load_model("pet_breed_classifier.keras")
with open("label_map.json") as f:
    idx2label = json.load(f)

IMAGE_SIZE   = (224, 224)
THRESHOLD    = 0.65    # top prob must be above this to be "known"
ENTROPY_MAX  = 1.5     # above this → unknown
TEMPERATURE  = 0.2    # softens overconfident softmax

# ── 2. INFERENCE FUNCTION ────────────────────────────────────────────────────
def predict(pil_image):
    if pil_image is None:
        return {}, ""

    # Preprocess — same pipeline as training
    img = pil_image.convert("RGB").resize((460, 460))
    img = img.crop((
        (460 - IMAGE_SIZE[0]) // 2,
        (460 - IMAGE_SIZE[1]) // 2,
        (460 + IMAGE_SIZE[0]) // 2,
        (460 + IMAGE_SIZE[1]) // 2,
    ))
    arr = np.array(img, dtype=np.float32)
    arr = keras.applications.resnet.preprocess_input(arr)
    arr = np.expand_dims(arr, 0)          # (1, 224, 224, 3)

    # Temperature scaling — softens overconfident softmax
    logits = model(arr, training=False)[0].numpy()
    logits = logits / TEMPERATURE
    probs  = np.exp(logits) / np.sum(np.exp(logits))

    top_prob = probs.max()
    entropy  = -np.sum(probs * np.log(probs + 1e-9))

    # ── Unknown breed detection ──────────────────────────────────────────────
    is_unknown = (top_prob < THRESHOLD) or (entropy > ENTROPY_MAX)

    top5_idx = np.argsort(probs)[::-1][:5]
    top5     = {
        idx2label[str(i)].replace("_", " ").title(): float(probs[i])
        for i in top5_idx
    }

    if is_unknown:
        message = (
            "🐾 **This breed is unknown to me.**\n\n"
            "Please contact the creator to add your breed if you wish.\n\n"
            "Closest known breeds shown below."
        )
    else:
        message = f"✅ Confidence: {top_prob*100:.1f}%"

    return top5, message


# ── 3. GRADIO UI ─────────────────────────────────────────────────────────────
with gr.Blocks(title="🐾 Pet Breed Classifier") as demo:

    gr.Markdown(
        """
        # 🐾 Pet Breed Classifier
        Upload a photo of a cat or dog and the model will predict the breed.
        Trained on the **Oxford-IIIT Pets** dataset (37 breeds).
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(type="pil", label="Upload pet photo")
            with gr.Row():
                submit_btn = gr.Button("🔍 Classify", variant="primary")
                clear_btn  = gr.ClearButton([image_input], value="🗑️ Clear")

        with gr.Column(scale=1):
            label_output   = gr.Label(num_top_classes=5, label="Top 5 Predictions")
            message_output = gr.Markdown(label="Status")

    submit_btn.click(
        fn=predict,
        inputs=image_input,
        outputs=[label_output, message_output]
    )
    image_input.change(
        fn=predict,
        inputs=image_input,
        outputs=[label_output, message_output]
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cf90871f9274b4d7bf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
